In [ ]:
from pathlib import Path
import subprocess, sys
import img2pdf
import numpy as np
import cv2
from PIL import Image, ImageOps
from tqdm import tqdm

# --- 設定 ---
INPUT_DIR    = Path("./data/sample1/")                # 画像のあるフォルダ（ファイル名昇順でページ順）
OUTPUT_PDF   = Path("./output/book_ocr.pdf")   # 最終PDF（検索可能）
TMP_DIR      = Path("./tmp")                   # 中間ファイル保存先（削除しない）
LANG         = "jpn+eng"                       # OCR言語
DPI          = 300                             # 画像に埋める論理DPI
USE_CUDA     = False                           # CUDA環境があれば True

# --- Docuwarp (UVDoc) ---
from docuwarp.unwarp import Unwarp

def load_uvdoc(use_cuda: bool) -> Unwarp:
    providers = ["CUDAExecutionProvider"] if use_cuda else None
    try:
        return Unwarp(providers=providers)
    except Exception as e:
        if use_cuda:
            print("[WARN] CUDA init failed, falling back to CPU:", e)
            return Unwarp(providers=None)
        raise

uvdoc = load_uvdoc(USE_CUDA)

def sorted_images(d: Path):
    exts = {".jpg",".jpeg",".png",".tif",".tiff",".bmp",".webp"}
    xs = [p for p in d.iterdir() if p.suffix.lower() in exts]
    xs.sort(key=lambda p: p.name)
    return xs

def enhance_after_dewarp(img_bgr: np.ndarray) -> np.ndarray:
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l,a,b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l2 = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l2,a,b]), cv2.COLOR_LAB2BGR)

def pil_read_rgb_with_exif(p: Path) -> Image.Image:
    im = Image.open(p).convert("RGB")
    return ImageOps.exif_transpose(im)

def uvdoc_dewarp_pil(im_rgb: Image.Image) -> Image.Image:
    try:
        return uvdoc.inference(im_rgb)
    except Exception as e:
        msg = str(e)
        if "Unexpected input data type" in msg and "int32" in msg and "int64" in msg:
            arr = np.array(im_rgb)
            arr = arr.astype(np.uint8, copy=False)
            return Image.fromarray(arr)
        raise

def main():
    pages = sorted_images(INPUT_DIR)
    if not pages:
        print(f"No images in {INPUT_DIR}")
        return

    OUTPUT_PDF.parent.mkdir(parents=True, exist_ok=True)
    TMP_DIR.mkdir(parents=True, exist_ok=True)

    dewarped_paths = []
    for p in tqdm(pages, desc="Dewarp & enhance"):
        try:
            im = pil_read_rgb_with_exif(p)
            im_unwarped = uvdoc_dewarp_pil(im)
        except Exception as e:
            print(f"[WARN] UVDoc dewarp failed on {p.name}: {e}")
            im_unwarped = pil_read_rgb_with_exif(p)  # フォールバック

        bgr = cv2.cvtColor(np.array(im_unwarped), cv2.COLOR_RGB2BGR)
        bgr = enhance_after_dewarp(bgr)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

        out_png = TMP_DIR / f"{p.stem}_dew.png"
        Image.fromarray(rgb).save(out_png, dpi=(DPI, DPI))
        dewarped_paths.append(out_png)

    # 画像 → 1つのPDF（img2pdf）
    interim_pdf = TMP_DIR / "images_merged.pdf"
    with open(interim_pdf, "wb") as f:
        f.write(img2pdf.convert([str(p) for p in dewarped_paths]))

    # OCRmyPDFで検索可能PDF化
    cmd = [
        sys.executable, "-m", "ocrmypdf",
        "--language", LANG,
        "--rotate-pages", "--deskew",
        "--optimize", "3",
        "--jobs", "auto",
        "--output-type", "pdfa",
        str(interim_pdf), str(OUTPUT_PDF)
    ]
    subprocess.run(cmd, check=True)
    print("Saved:", OUTPUT_PDF.resolve())
    print("Temporary files kept in:", TMP_DIR.resolve())

if __name__ == "__main__":
    main()
